In [1]:
%pip install optbinning pandas xlrd duckdb



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from itertools import combinations
from optbinning import OptimalBinning


In [3]:
data = pd.read_excel("default of credit card clients.xls", header=1)
data = data.drop(columns=["ID"])
target = "default payment next month"
data.head()


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [4]:
def target_value_counts(df, target_col, normalize=True, pct=True):
    counts = df[target_col].value_counts(normalize=normalize)
    return counts * 100 if pct else counts

def compute_iv_ranking(df, target, solver="cp"):
    results = []
    for col in df.columns:
        if col == target:
            continue
        try:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype, solver=solver)
            optb.fit(df[col].values, df[target].values)
            iv = optb.binning_table.build().IV.iloc[-1] 
            results.append({"variable": col, "iv": iv * 100})
        except Exception as exc:
            print(f"Skipping {col}: {exc}")
    return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

def create_binned_df(df, target, variables):
    binned_df = pd.DataFrame(index=df.index)
    for col in variables:
        dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(df[col].values, df[target].values)
        binned_df[col] = optb.transform(df[col], metric="bins")
    binned_df[target] = df[target].values
    return binned_df


In [5]:
def one_way_summary(binned_df, target, base_rate, pct=True):
    results = []
    for col in binned_df.columns:
        if col == target:
            continue
        summary = (
            binned_df
            .groupby(col)
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = col + "=" + summary[col].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def two_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2 in combinations(cols, 2):
        summary = (
            binned_df
            .groupby([c1, c2])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = c1 + "=" + summary[c1].astype(str) + sep + c2 + "=" + summary[c2].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def three_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2, c3 in combinations(cols, 3):
        summary = (
            binned_df
            .groupby([c1, c2, c3])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = (
            c1 + "=" + summary[c1].astype(str) + sep +
            c2 + "=" + summary[c2].astype(str) + sep +
            c3 + "=" + summary[c3].astype(str)
        )
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def shortlist_rules(df, min_sample_size=1000, min_lift=2.0, base_rate=None):
    if base_rate is None:
        raise ValueError("base_rate is required")
    threshold = base_rate * 100
    return df.assign(
        shortlist=(
            (df["count"] >= min_sample_size) &
            (df["lift"] >= min_lift) &
            (df["rate"] > threshold)
        ).astype(int)
    )


In [6]:
iv_ranking = compute_iv_ranking(data, target)
top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(data, target, top_vars)
base_rate = data[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=...",1002,77.245509,3.492112,1
1,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf) & PAY_6=...",1072,77.238806,3.491809,1
2,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_6=...",1091,77.176902,3.489010,1
3,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf) & PAY_6=...",1024,76.855469,3.474479,1
4,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1249,76.621297,3.463892,1
5,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1053,76.448243,3.456069,1
6,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_5=...",1181,76.291279,3.448973,1
7,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1000,76.200000,3.444846,1
8,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1309,76.088617,3.439811,1
9,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1011,75.964392,3.434195,1


In [7]:
def parse_rule(rule_str):
    parts = [p.strip() for p in rule_str.split("&")]
    conditions = []
    for part in parts:
        column, interval = part.split("=", 1)
        column = column.strip()
        interval = interval.strip()
        include_lower = interval.startswith("[")
        include_upper = interval.endswith("]")
        lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
        def to_bound(value):
            value = value.lower()
            if value in {"inf", "infty", "+inf", "+infty"}:
                return float("inf")
            if value in {"-inf", "-infty"}:
                return float("-inf")
            return float(value)
        conditions.append({
            "column": column,
            "lower": to_bound(lower_str),
            "upper": to_bound(upper_str),
            "include_lower": include_lower,
            "include_upper": include_upper,
        })
    return conditions

def build_mask(df, conditions):
    mask = pd.Series(True, index=df.index)
    for cond in conditions:
        column = cond["column"]
        column_values = pd.to_numeric(df[column], errors="coerce")
        current = pd.Series(True, index=df.index)
        if cond["include_lower"]:
            current &= column_values >= cond["lower"]
        else:
            current &= column_values > cond["lower"]
        if cond["include_upper"]:
            current &= column_values <= cond["upper"]
        else:
            current &= column_values < cond["upper"]
        mask &= current.fillna(False)
    return mask

first_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(first_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100


default payment next month
0    79.784813
1    20.215187
Name: proportion, dtype: float64

In [ ]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


In [ ]:
second_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(second_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    80.534485
1    19.465515
Name: proportion, dtype: float64

In [ ]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf)",1094,69.652651,3.578259,1
1,"PAY_0=[1.50, inf) & SEX=(-inf, 1.50)",1362,69.016153,3.545560,1
2,"PAY_0=[1.50, inf)",1606,67.496887,3.467511,1
3,"PAY_2=[1.50, inf) & PAY_5=[1.00, inf) & PAY_6=...",1007,66.534260,3.418058,1
4,"PAY_2=[1.50, inf) & PAY_3=[1.50, inf) & PAY_6=...",1026,65.692008,3.374789,1
5,"PAY_2=[1.50, inf) & PAY_4=[0.50, inf) & PAY_5=...",1111,64.176418,3.296929,1
6,"PAY_2=[1.50, inf) & PAY_3=[1.50, inf) & SEX=(-...",1335,64.044944,3.290175,1
7,"PAY_2=[1.50, inf) & PAY_6=[1.00, inf)",1209,64.019851,3.288886,1
8,"PAY_2=[1.50, inf) & PAY_3=[1.50, inf) & PAY_5=...",1118,63.864043,3.280881,1
9,"PAY_2=[1.50, inf) & PAY_5=[1.00, inf)",1283,62.821512,3.227323,1


In [ ]:
third_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(third_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    81.65895
1    18.34105
Name: proportion, dtype: float64

In [ ]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[0.50, inf) & PAY_3=[-0.50, 1.50)",1730,51.502890,2.808067,1
1,"PAY_0=[0.50, inf) & PAY_3=[-0.50, 1.50) & PAY_...",1296,49.074074,2.675641,1
2,"PAY_0=[0.50, inf) & PAY_3=[-0.50, 1.50) & PAY_...",1534,48.891786,2.665703,1
3,"PAY_0=[0.50, inf) & PAY_3=[-0.50, 1.50) & PAY_...",1354,48.744461,2.657670,1
4,"PAY_0=[0.50, inf) & BILL_AMT1=[782.50, 19986.50)",1055,48.246445,2.630517,1
5,"PAY_5=[1.00, inf) & PAY_6=[1.00, inf)",1133,47.219771,2.574540,1
6,"PAY_4=[0.50, inf) & PAY_5=[1.00, inf)",1153,47.181266,2.572441,1
7,"PAY_2=[1.50, inf) & PAY_3=[1.50, inf)",1195,47.112971,2.568717,1
8,"PAY_0=[0.50, inf) & PAY_3=[1.50, inf)",1098,46.994536,2.562260,1
9,"PAY_0=[0.50, inf) & PAY_4=[-0.50, 0.50) & EDUC...",1004,46.613546,2.541487,1


In [ ]:
forth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(forth_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    80.213215
1    19.786785
Name: proportion, dtype: float64

In [ ]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[1.50, inf) & PAY_3=[1.00, inf) & PAY_6=...",1024,76.855469,3.884182,1
1,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_6=...",1026,76.803119,3.881536,1
2,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1031,76.624636,3.872516,1
3,"PAY_0=[1.50, inf) & PAY_4=[1.00, inf) & PAY_5=...",1105,75.927602,3.837288,1
4,"PAY_0=[1.50, inf) & PAY_3=[1.00, inf) & PAY_5=...",1108,75.722022,3.826899,1
5,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_5=...",1110,75.675676,3.824556,1
6,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1115,75.515695,3.816471,1
7,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_4=...",1348,73.961424,3.737920,1
8,"PAY_0=[1.50, inf) & PAY_3=[1.00, inf) & PAY_4=...",1351,73.945226,3.737102,1
9,"PAY_0=[1.50, inf) & PAY_4=[1.00, inf)",1362,73.715125,3.725473,1


In [ ]:
fifth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(fifth_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    79.814329
1    20.185671
Name: proportion, dtype: float64

In [ ]:
import numpy as np

In [ ]:
def convert_interval_to_query(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a pandas .query() compatible string.
    """
    query_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column
        if col_conditions:
            query_conditions.append(" & ".join(col_conditions))
            
    # Combine all column rules into the final query string
    return " & ".join(query_conditions)

# --- Test the converter ---
input_string = first_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=[1.00, inf)
Output: PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00


In [ ]:
# --- Test the converter ---
input_string = second_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & BILL_AMT4=[5756.00, inf) & SEX=[1.50, inf)
Output: PAY_0 >= 1.50 & BILL_AMT4 >= 5756.00 & SEX >= 1.50


In [ ]:
# --- Test the converter ---
input_string = third_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & PAY_2=[1.50, inf)
Output: PAY_0 >= 1.50 & PAY_2 >= 1.50


In [ ]:
# --- Test the converter ---
input_string = forth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[0.50, inf) & PAY_3=[-0.50, 1.50)
Output: PAY_0 >= 0.50 & PAY_3 >= -0.50 & PAY_3 < 1.50


In [ ]:
# --- Test the converter ---
input_string = fifth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & PAY_3=[1.00, inf) & PAY_6=[1.00, inf)
Output: PAY_0 >= 1.50 & PAY_3 >= 1.00 & PAY_6 >= 1.00


In [ ]:
data['default payment next month'].value_counts(normalize=True) * 100

default payment next month
0    77.88
1    22.12
Name: proportion, dtype: float64

In [ ]:
data.query("PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00")[target].value_counts(normalize=True) * 100

default payment next month
1    77.245509
0    22.754491
Name: proportion, dtype: float64

In [ ]:
subset_data = data[~data.index.isin(data.query("PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00").index)]

In [ ]:
subset_data.query("PAY_0 >= 1.50 & BILL_AMT4 >= 5756.00 & SEX >= 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    69.230769
0    30.769231
Name: proportion, dtype: float64

In [ ]:
data.query("PAY_0 >= 1.50 & PAY_2 >= 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    71.341748
0    28.658252
Name: proportion, dtype: float64

In [ ]:
data.query("PAY_0 >= 0.50 & PAY_3 >= -0.50 & PAY_3 < 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    53.981436
0    46.018564
Name: proportion, dtype: float64

In [ ]:
import duckdb
result = duckdb.sql("""

SELECT CASE WHEN PAY_0 >= 1.50 AND PAY_4 >= 0.50 AND PAY_6 >= 1.00 THEN 1
WHEN PAY_0 >= 1.50 AND BILL_AMT4 >= 5756.00 AND SEX >= 1.50 THEN 2
WHEN PAY_0 >= 1.50 AND PAY_2 >= 1.50 THEN 3
WHEN PAY_0 >= 0.50 AND PAY_3 >= -0.50 AND PAY_3 < 1.50 THEN 4
WHEN PAY_0 >= 1.50 AND PAY_3 >= 1.00 AND PAY_6 >= 1.00 THEN 5
ELSE 0 END AS segment, 
COUNT(*) AS count,
SUM("default payment next month") AS events,
(SUM("default payment next month") / COUNT(*)) * 100 AS rate,
FROM data
GROUP BY segment

""").df()


In [ ]:
result

,segment,count,events,rate
0,0,26113,4208.0,16.114579
1,1,1002,774.0,77.245509
2,2,1040,720.0,69.230769
3,3,601,380.0,63.227953
4,4,1243,553.0,44.489139
5,5,1,1.0,100.000000
